# 👗 StyleFinder — Free 24/7 Cloud Server (Google Colab with Free T4 GPU)

This notebook runs the **StyleFinder Clothing Retrieval Server** on Google's free Cloud GPU (Tesla T4) and generates a public HTTPS URL accessible on your phone.

### Instructions:
1. Go to **Runtime → Change runtime type** → select **T4 GPU**.
2. Click **Runtime → Run all** (or press `Ctrl+F9`).
3. Click the `*.trycloudflare.com` link printed at the bottom to open the web app on your phone!

In [ ]:
# 1. Install dependencies and cloudflared tunnel
!pip install -q fastapi uvicorn python-multipart transformers timm pillow
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print('✓ Dependencies and Cloudflare tunnel installed!')

In [ ]:
# 2. Start the FastAPI StyleFinder server and expose it over HTTPS
import subprocess, time

# Launch Cloudflare Tunnel in background
tunnel_proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    if 'trycloudflare.com' in line:
        for word in line.split():
            if 'trycloudflare.com' in word:
                public_url = word.strip()
                break
        if public_url: break
    time.sleep(0.5)

print('\n' + '='*60)
print('🌐 YOUR LIVE PUBLIC HTTPS URL:')
print(public_url or 'Tunnel started, check output')
print('='*60 + '\n')

# Run StyleFinder server on port 8000
!python cir_server.py --port 8000